# Dense-width curves and additional controls

Retain the stationary learning curves across all seven dense widths, the non-stationary curves, and the permutation-interval control with a wider dense network. These experiments are not in the current paper.

The stationary step-sizes maximize final accuracy rather than early learning speed.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from multi_mnist.analysis import (
    fetch_per_seed_arrays, filter_sweep, get_color_palette, label_curves_inline,
    load_export, normalize_columns, plot_ci_curve, save_fig, set_style, stack_per_seed,
)

%matplotlib inline

FIG_DIR = '../figures/generated'
PALETTE = get_color_palette(classes=['Block-Sparse', 'Dense'])

In [ ]:
# Original MLflow exports, one child run per seed.
OLD_DATA = Path('../../../phd/structure_search/analysis/weight_pruning/data')
params = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_params.csv',
                     index_col=0, low_memory=False)


## Permutation intervals with a wider dense network

Compare block-sparse at 24 hidden units with dense at 48 units. Each point uses 5 seeds and the step-size with the highest final-10% accuracy.

The original runs apply permutation events at the start of each 1000-step log window. Thus, short intervals produce bursts of permutations.


In [ ]:
# Run once to fetch the rebuttal runs from Comet, then leave commented out.
# from multi_mnist.analysis import download_project
# download_project('paper-weight-pruning-rebuttal', output_dir='data')
new_params, new_metrics = load_export('paper-weight-pruning-rebuttal', data_dir='data')

# Pre-release sweeps. Parent runs carry the 5-seed mean curve.
old_params = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_sweep_params.csv',
                         index_col=0, low_memory=False)
old_params = old_params[old_params['child_seed'].isna()]
old_metrics = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_sweep_metrics.csv',
                          usecols=['run_id', 'step', 'accuracy'])
N_PERIODS = 500   # 500,000 steps logged every 1000


def last_tenth(acc):
    """Mean accuracy over a run's last 10% of steps; NaN if it diverged and stopped early."""
    acc = acc.dropna()
    return acc.iloc[-N_PERIODS // 10:].mean() if len(acc) == N_PERIODS else np.nan


def best_per_period(params, metrics):
    """Each permutation period's accuracy at its best step-size."""
    acc = (metrics[metrics['run_id'].isin(params['run_id'])]
           .groupby('run_id')['accuracy'].apply(last_tenth))
    cells = params.assign(acc=params['run_id'].map(acc))
    return cells.groupby(pd.to_numeric(cells['task.permute_period']))['acc'].max()


block_sparse = old_params[old_params['sweep_name'] == '04_block_sparse_permute_period_lr']
dense_800 = old_params[(old_params['sweep_name'] == '02_dense_scaling_4task_nonstationary_lr')
                       & (old_params['model.initial_hidden_units'] == 48)]
dense_rest = new_params[new_params['sweep_name'] == '05_dense_8x_permute_period']
PERMUTE = {  # label: (accuracy per period, color)
    'Block-Sparse': (best_per_period(block_sparse, old_metrics), PALETTE['Block-Sparse']),
    'Dense (8×)': (pd.concat([best_per_period(dense_800, old_metrics),
                              best_per_period(dense_rest, new_metrics)]).sort_index(),
                   PALETTE['Dense']),
}

set_style('2-col')
fig, ax = plt.subplots()
for label, (acc, color) in PERMUTE.items():
    ax.plot(acc.index, acc.values, '-o', label=label, color=color)
periods = PERMUTE['Block-Sparse'][0].index
ax.set_xscale('log', base=2); ax.minorticks_off()
ax.set_xticks(periods); ax.set_xticklabels([f'{p:.0f}' for p in periods])
ax.set_ylim(0.6, 1.0)
ax.set_xlabel('Steps Between Permutations'); ax.set_ylabel('Accuracy')
ax.set_title('Permutation Rate')
ax.grid(True); ax.set_axisbelow(True)
ax.legend(loc='lower right')
save_fig('rebuttal_permute_period', fig_dir=FIG_DIR)
plt.show()

## Stationary learning curves across all dense widths

Compare block-sparse at 24 units with dense at 6–384 units. Each curve uses 30 seeds and its original step-size. The first 25,000 steps show early learning. Step-sizes were selected for final accuracy.


In [ ]:
WINDOW_2A = (0, 25_000)   # the first 5% of 500,000 steps
FIG2A = [('Block-Sparse', '01_block_sparse_4task_best', 24, PALETTE['Block-Sparse'])] + [
    (f'Dense ({2**i}×)', '01_dense_scaling_4task_best', units, color)
    for i, (units, color) in enumerate(zip([6, 12, 24, 48, 96, 192, 384],
                                           plt.cm.Reds(np.linspace(0.3, 1.0, 7))))]

# Figure 2a's runs are in the same pre-release export as Plot 1's.
metrics_2a = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_metrics.csv',
                         usecols=['run_id', 'step', 'loss', 'accuracy'])
metrics_2a = metrics_2a[metrics_2a['loss'].notna() & metrics_2a['step'].between(*WINDOW_2A)]

set_style('2-col')
fig, ax = plt.subplots()
for label, sweep, units, color in FIG2A:
    runs = params[(params['sweep_name'] == sweep) & params['child_seed'].notna()
                  & (params['model.initial_hidden_units'] == units)]
    accs = (metrics_2a[metrics_2a['run_id'].isin(runs['run_id'])]
            .pivot(index='step', columns='run_id', values='accuracy'))
    plot_ci_curve(accs.index, accs.to_numpy(), label=label, color=color, ax=ax)

ax.set_xlim(*WINDOW_2A); ax.set_ylim(0, 1.0)
ax.set_xlabel('Step'); ax.set_ylabel('Accuracy')
ax.set_title('Stationary, First 5%')
ax.grid(True); ax.set_axisbelow(True)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
save_fig('rebuttal_stationary_scaling_first_5pct', fig_dir=FIG_DIR)
plt.show()

In [ ]:
# Non-stationary learning curves over the final 6,000 steps, from the `_high_freq` reruns,
# which log every 100 steps (the finest any of these runs were logged at).
WINDOW_NS = (494_000, 500_000)   # the last 6,000 of 500,000 steps
FIG2_NS = [('Block-Sparse', '02_block_sparse_4task_nonstationary_best_high_freq', 24, PALETTE['Block-Sparse'])] + [
    (label, '02_dense_scaling_4task_nonstationary_best_high_freq', units, color)
    for (label, _, units, _), color in zip(FIG2A[3:], plt.cm.Reds(np.linspace(0.3, 1.0, 5)))]

metrics_ns = pd.read_csv(OLD_DATA / 'paper_weight_pruning_scaling_best_metrics.csv',
                         usecols=['run_id', 'step', 'loss', 'accuracy'])
metrics_ns = metrics_ns[metrics_ns['loss'].notna() & metrics_ns['step'].between(*WINDOW_NS)]

set_style('2-col')
fig, ax = plt.subplots()
for label, sweep, units, color in FIG2_NS:
    runs = params[(params['sweep_name'] == sweep) & params['child_seed'].notna()
                  & (params['model.initial_hidden_units'] == units)]
    accs = (metrics_ns[metrics_ns['run_id'].isin(runs['run_id'])]
            .pivot(index='step', columns='run_id', values='accuracy'))
    plot_ci_curve(accs.index, accs.to_numpy(), label=label, color=color, ax=ax)

ax.set_xlim(*WINDOW_NS); ax.set_ylim(0.55, 0.95)
ax.set_xlabel('Step'); ax.set_ylabel('Accuracy')
ax.set_title('Non-Stationary, Last 6K')
ax.grid(True); ax.set_axisbelow(True)
ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0), useMathText=True, useOffset=False)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
save_fig('rebuttal_nonstationary_scaling_last_5pct', fig_dir=FIG_DIR)
plt.show()